In [ ]:
import os

# Konfiguracja API Kaggle
os.environ['KAGGLE_USERNAME'] = "sikoraa36" 
os.environ['KAGGLE_KEY'] = "KLUCZ_API" 
# !kaggle datasets download -d krzysztofjamroz/apartment-prices-in-poland --unzip

In [ ]:
import pandas as pd
import glob

# 1. Szukamy wszystkich plików ze sprzedażą (zaczynają się od "apartments_pl_") Ignorujemy pliki "rent" (wynajem)
files = sorted(glob.glob("apartments_pl_*.csv"))

print(f"Znaleziono {len(files)} plików do połączenia.")

# 2. Wczytujemy je po kolei i dodajemy do listy
dfs = []
for filename in files:
    temp_df = pd.read_csv(filename)
    # Wyciągamy rok i miesiąc z nazwy pliku, żeby wiedzieć kiedy była oferta
    # np. z 'apartments_pl_2023_09.csv' robimy '2023_09'
    month_id = filename.replace("apartments_pl_", "").replace(".csv", "")
    temp_df['year_month'] = month_id 
    dfs.append(temp_df)

# 3. Sklejamy w jeden wielki DataFrame
df = pd.concat(dfs, ignore_index=True)

print(f"SUKCES! Łączna liczba ofert w Twojej bazie: {df.shape[0]}")

# Wyświetl losowe 5 ofert, żeby zobaczyć jak wyglądają dane
display(df.sample(5))

In [ ]:
# Sprawdzamy procent brakujących danych w każdej kolumnie
missing_data = df.isnull().mean() * 100
# tylko te kolumny, gdzie faktycznie czegoś brakuje (więcej niż 0%)
missing_data = missing_data[missing_data > 0].sort_values(ascending=False)
print("Procent brakujących danych w kolumnach: ")
print(missing_data)

In [ ]:
def clean_data(df):
    df_clean = df.copy()
    
    # 1. USUWANIE KOLUMN ze zbyt dużą liczbą braków
    cols_to_drop = ['condition', 'buildingMaterial', 'type']
    df_clean = df_clean.drop(columns=cols_to_drop, errors='ignore')
    print(f"Usunięto kolumny: {cols_to_drop}")

    # 2. UZUPEŁNIANIE DANYCH
    
    # hasElevator: brak danych = brak windy (0)
    # Najpierw wypełniamy, potem wymuszamy typ liczbowy
    df_clean['hasElevator'] = df_clean['hasElevator'].fillna(0)
    
    # Odległości (Distance): Wypełniamy medianą
    distance_cols = [col for col in df_clean.columns if 'Distance' in col]
    for col in distance_cols:
        median_value = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_value)
    print("Uzupełniono braki w windzie i odległościach.")

    # 3. USUWANIE WIERSZY w kluczowych kolumnach
    # Jeśli nie znamy piętra lub roku budowy to odrzucamy ofertę
    critical_cols = ['floor', 'buildYear', 'floorCount']
    before_rows = df_clean.shape[0]
    df_clean = df_clean.dropna(subset=critical_cols)
    after_rows = df_clean.shape[0]
    
    print(f"Usunięto wiersze z brakami w kluczowych danych.")
    print(f"Liczba wierszy przed: {before_rows}")
    print(f"Liczba wierszy po:   {after_rows}")
    print(f"Straciliśmy: {before_rows - after_rows} ofert ({(before_rows - after_rows)/before_rows:.1%} zbioru)")
    
    return df_clean

df = clean_data(df)

print("\nSprawdzenie końcowe:")
print(df.isnull().sum().sum()) # Jeśli wynik to 0, to jest idealnie!

In [ ]:
display(df.describe())

In [ ]:
# Filtrujemy tylko bardzo stare budynki (sprzed 1900 roku)
stare_kamienice = df[df['buildYear'] < 1900]

# Sprawdźmy, w jakich miastach występują najczęściej
print("\n--- Gdzie są te stare budynki? (Top 5 miast) ")
print(stare_kamienice['city'].value_counts().head(5))

# Zobaczmy przykładowe oferty (Cena za m2 pokaże nam, czy to ruina czy luksus)
display(stare_kamienice[['city', 'buildYear', 'price', 'squareMeters']].sort_values(by='buildYear', ascending=True).head(20))


In [ ]:
# Tworzymy kolumnę price_per_m2 (Cena całkowita / Metraż)
df['price_per_m2'] = df['price'] / df['squareMeters']

# Sprawdzamy mieszkania powyżej 20. piętra
wysokie_pietra = df[df['floor'] > 20]

# W jakich miastach się znajdują
print("\nGdzie są wieżowce?")
print(wysokie_pietra['city'].value_counts())

# Spójrzmy na ceny - powinny być BARDZO wysokie (to luksusowe apartamenty)
display(wysokie_pietra[['city', 'floor', 'price_per_m2', "squareMeters"]].sort_values(by='floor', ascending=False).head(15))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Wybieramy tylko kolumny liczbowe
df_numeric = df.select_dtypes(include=['float64', 'int64'])

# Obliczamy korelację
correlation_matrix = df_numeric.corr()

# Heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title("Macierz Korelacji")
plt.show()

print("Co najbardziej podbija cenę za m2?")
# Sortujemy korelacje względem ceny za m2
print(correlation_matrix['price_per_m2'].sort_values(ascending=False))

In [ ]:
# Wybieramy tylko te kolumny, które chcemy dać modelowi
# Odrzucamy te mało istotne (id) oraz te, które zdradzają odpowiedź (price - bo przewidujemy cenę)
features = [
    'city', 'squareMeters', 'rooms', 'floor', 'floorCount', 'buildYear',
    'latitude', 'longitude', 'centreDistance', 'poiCount', 'schoolDistance',
    'clinicDistance', 'pharmacyDistance', 'kindergartenDistance', 'restaurantDistance',
    'hasElevator' # hasElevator to 0 lub 1, więc jest ok
]

X = df[features]
y = df['price'] # Target

# Zamiana miast na bool (One-Hot Encoding)
# Zmiana 'city' na 'city_Warszawa', 'city_Wroclaw' itd.
X = pd.get_dummies(X, columns=['city'], dtype=int)

print("Gotowe dane do modelu")
print(f"Liczba kolumn po transformacji: {X.shape[1]}")
display(X.head())